<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [1]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
  subset='train',
  remove=('headers', 'footers', 'quotes')
).data[:3000]

In [2]:
type(docs)

list

In [6]:
docs[10]

'I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs\nvery well, paint is the bronze/brown/orange faded out, leaks a bit of oil\nand pops out of 1st with hard accel.  The shop will fix trans and oil \nleak.  They sold the bike to the 1 and only owner.  They want $3495, and\nI am thinking more like $3K.  Any opinions out there?  Please email me.\nThanks.  It would be a nice stable mate to the Beemer.  Then I\'ll get\na jap bike and call myself Axis Motors!\n\n-- \n-----------------------------------------------------------------------\n"Tuba" (Irwin)      "I honk therefore I am"     CompuTrac-Richardson,Tx\nirwin@cmptrc.lonestar.org    DoD #0826          (R75/6)'

# STEP 1 – **Preprocessing**

In [ ]:
import re

def preprocess(text):
  text = re.sub(r'[^a-z\s]', '', text.lower())
  return text.split()


# STEP 2 – **Vocabulary**

In [ ]:
#comlete the code
def build_vocab(docs):
  vocab = set()
  for doc in docs:
    for token in preprocess(doc):
      vocab.add(token)
  return sorted(list(vocab))


# STEP 3 – **DF**

In [ ]:
def compute_df(docs, vocab):
  df = {word: 0 for word in vocab}
  for doc in docs:
    tokens = set(preprocess(doc))
    for word in tokens:
      if word in df:
        df[word] += 1
  return df


# STEP 4 – **IDF**

In [ ]:
def compute_idf(df, N):
  import math
  idf = {}
  for word, d_freq in df.items():
    idf[word] = math.log((N + 1) / (d_freq + 1)) + 1
  return idf


# STEP 5 – **TF Vector**

In [ ]:
def compute_tf(doc, vocab):
  tokens = preprocess(doc)
  if not tokens:
    return [0.0] * len(vocab)
  total = len(tokens)
  counts = {}
  for t in tokens:
    counts[t] = counts.get(t, 0) + 1
  return [counts.get(word, 0) / total for word in vocab]


# STEP 6 – **TF-IDF Matrix**

In [ ]:
def build_tfidf(docs):
  vocab = build_vocab(docs)
  N = len(docs)
  df = compute_df(docs, vocab)
  idf = compute_idf(df, N)
  
  tfidf_matrix = []
  for doc in docs:
    tf = compute_tf(doc, vocab)
    tfidf_vec = [tf_val * idf[word] for tf_val, word in zip(tf, vocab)]
    tfidf_matrix.append(tfidf_vec)
  
  return tfidf_matrix, vocab


# STEP 7 – **Cosine Similarity**

In [13]:
def cosine(a, b):
  import numpy as np
  a = np.array(a, dtype=float)
  b = np.array(b, dtype=float)
  dot = np.dot(a, b)
  norm_a = np.linalg.norm(a)
  norm_b = np.linalg.norm(b)
  if norm_a == 0 or norm_b == 0:
    return 0.0
  return dot / (norm_a * norm_b)

# STEP 8 – **Search Engine**

In [ ]:
import numpy as np
def search(query, docs, top_k=5):
  tfidf_matrix, vocab = build_tfidf(docs)

  q_vec = np.zeros(len(vocab)) # build query vector
  q_words = preprocess(query)

  for i, w in enumerate(vocab):
    q_vec[i] = q_words.count(w)

  if np.sum(q_vec) > 0:
    q_vec = q_vec / np.sum(q_vec)

  scores = []

  for i, doc_vec in enumerate(tfidf_matrix):
    score = cosine(q_vec, doc_vec)
    scores.append((score, i))

  scores.sort(reverse=True)

  return scores[:top_k]

> TEST

In [15]:
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
  print(score)
  print(docs[idx][:200])
  print("-" * 50)

0.18901623997706526
I just recently bought a 4 MB ram card for my original mac portable 
(backlit) and have since had some bizarre crashes. It happens when I put 
the machine to sleep and wake the machine up. sometimes i
--------------------------------------------------
0.17333946230553898
I'm looking for a Singer Featherweight 221 sewing machine (old, black 
sewing machine in black case).

Please contact:
--------------------------------------------------
0.1413719690773178

Yeah.  Sounds typical.  Windows makes all sorts of extra demands on hardware,
and therefore your machine can't keep up with things.  Ever notice how when
acessing the floppies in Windows, everything 
--------------------------------------------------
0.12876135761937252



The previous article referred to the fact that you could only use 20ns SIMMs in
a 50MHz machine, but that you could use 80ns SIMMs in slower machines. I just
pointed out that if you could only use 
----------------------------------------------